# Lecture 3: Password Attack with Differential-Power-Analysis (Kocher et al. 1999)

In [1]:
%load_ext autoreload
%autoreload 2

import os
import random

import numpy as np
import plotly.graph_objects as pgo
from cwtoolbox import CaptureDevice

In [2]:
capture_device = CaptureDevice.create("CWLITEXMEGA")
capture_device.compile(file=os.path.abspath("sbox_lookup.c"))
capture_device.flash()

c:\work\securecoding_ws2526\.venv\Lib\site-packages\chipwhisperer\capture\trace\TraceWhisperer.py:31: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources # type: ignore


XMEGA Programming flash...
XMEGA Reading flash...
Verified flash OK, 2553 bytes


In [3]:
data = capture_device.capture(
    number_of_traces=1000, input=lambda _: [random.randint(0, 255)] + 15 * [0]
)

fig = pgo.Figure()
for d in data[:10]:
    fig.add_trace(pgo.Scatter(y=d["trace"]))
fig.show()

100%|██████████| 1000/1000 [00:19<00:00, 51.90it/s]


In [4]:
data_lsb_0 = np.array([d for d in data if d["input"][0] & 0x80 == 0])
data_lsb_1 = np.array([d for d in data if d["input"][0] & 0x80 != 0])

In [5]:
mean_trace_lsb_0 = np.mean(data_lsb_0["trace"], axis=0)
mean_trace_lsb_1 = np.mean(data_lsb_1["trace"], axis=0)

mean_trace_lsb_0.shape, mean_trace_lsb_1.shape

((1376,), (1376,))

In [6]:
fig = pgo.Figure()
fig.add_trace(pgo.Scatter(y=mean_trace_lsb_0 - mean_trace_lsb_1))
fig.show()

In [7]:
from lascar.tools.aes import sbox


def split(arr, condition):
    return arr[np.array(condition)], arr[~np.array(condition)]

In [8]:
diff1_1, diff1_2 = split(data, [sbox[d["input"][0] ^ 0x00] & 0x80 == 0 for d in data])
diff2_1, diff2_2 = split(data, [sbox[d["input"][0] ^ 0x01] & 0x80 == 0 for d in data])

fig = pgo.Figure()
fig.add_trace(
    pgo.Scatter(
        y=np.abs(np.mean(diff1_1["trace"], axis=0) - np.mean(diff1_2["trace"], axis=0))
    )
)
fig.add_trace(
    pgo.Scatter(
        y=np.abs(np.mean(diff2_1["trace"], axis=0) - np.mean(diff2_2["trace"], axis=0))
    )
)
fig.show()

In [9]:
def aes_sbox_dpa(
    data,
    key_byte_index=0,
    selection_bit_index=0,
):
    diffs = []
    for guess in range(256):
        d1, d2 = split(
            data,
            [
                sbox[d["input"][key_byte_index] ^ guess] & (1 << selection_bit_index)
                == 0
                for d in data
            ],
        )
        diffs.append(
            (
                np.max(
                    np.abs(np.mean(d1["trace"], axis=0) - np.mean(d2["trace"], axis=0))
                ),
                guess,
            )
        )
    return sorted(diffs, reverse=True)

In [10]:
aes_sbox_dpa(data, key_byte_index=0, selection_bit_index=7)

[(np.float64(0.014318728898649546), 1),
 (np.float64(0.011863118212641355), 62),
 (np.float64(0.01030124098265428), 133),
 (np.float64(0.010279823414176159), 244),
 (np.float64(0.010195096823638922), 104),
 (np.float64(0.009185546874999978), 29),
 (np.float64(0.008903645770663854), 212),
 (np.float64(0.008740234375000017), 124),
 (np.float64(0.00848025287154594), 25),
 (np.float64(0.008376805471246973), 65),
 (np.float64(0.00823495768514737), 116),
 (np.float64(0.008054804981203034), 52),
 (np.float64(0.008054149114752718), 229),
 (np.float64(0.008043801706456827), 145),
 (np.float64(0.00790944858751036), 209),
 (np.float64(0.007882065507974573), 34),
 (np.float64(0.007864113487703939), 87),
 (np.float64(0.007788253809265233), 186),
 (np.float64(0.007764253273838118), 205),
 (np.float64(0.007639815715512865), 36),
 (np.float64(0.007607574160853087), 43),
 (np.float64(0.007513784405089818), 240),
 (np.float64(0.007395040793497065), 190),
 (np.float64(0.0073611545453226845), 250),
 (np.f